# How the DINOv3 pipeline localizes the head

Head localization in the `dinov3/` folder happens in two stages, then a 3-D fuse:

1. **`top3_head_patches.py`** — score every patch in each of the 6 canonical views by cosine similarity to an annotated "head" prototype, and keep the top-$k$ (default 3) candidate patches per view.
2. **`fuse_head_position.py`** — turn each candidate patch into a 3-D camera ray, then find the single 3-D point most consistent with the rays. This is done robustly with **RANSAC** wrapped around an **unweighted least-squares ray triangulation**.

The math below is the core of stage 2 — a plain (unweighted) least-squares triangulation embedded in a **RANSAC** outlier-rejection loop. Patch-similarity scores are still computed and used by RANSAC to rank hypotheses and select views, but the final geometric solve treats every inlier ray equally.

---

## Stage 1 — per-view patch scoring (`top3_head_patches.py`)

For one species and one view angle, the code builds a **head prototype** from the annotated reference set:

- Each annotated individual contributes the DINOv3 patch token at its known head patch $(\text{row},\text{col})$.
- These tokens are averaged into one prototype vector
  $$
  \mathbf{q}_{\text{proto}} = \frac{1}{N}\sum_{n=1}^{N}\mathbf{t}^{(n)}_{\text{head}}.
  $$
- Optionally a PCA fitted on the foreground tokens projects both prototype and query tokens into a lower-dimensional space (`--explained-variance`).

For a query image, every patch token $\mathbf{t}_j$ and the prototype are L2-normalized, and the similarity grid is just the dot product:
$$
s_j = \frac{\mathbf{t}_j}{\lVert\mathbf{t}_j\rVert}\cdot\frac{\mathbf{q}_{\text{proto}}}{\lVert\mathbf{q}_{\text{proto}}\rVert}\in[-1,1].
$$

`pick_top_k_peaks` then walks the foreground patches in descending similarity and greedily keeps peaks that are at least `--min-patch-distance` apart (default 2 grid cells), so the top-3 are spatially distinct rather than three adjacent cells of one blob. Each kept patch is written out with its pixel-space center, normalized to $[0,1]$ as `patch_center_x_norm` / `patch_center_y_norm` — these are the inputs to stage 2.

---

## Stage 2 — the geometry: distance from a point to a ray

Each view gives a ray $i$: the set $\{\,\mathbf{o}_i + t\,\mathbf{d}_i : t\in\mathbb{R}\,\}$, where $\mathbf{o}_i$ is the camera position and $\mathbf{d}_i$ is the unit direction toward the predicted head patch (`build_ray_from_pixel` un-projects the normalized patch center through the camera's FoV and basis; the code normalizes $\mathbf{d}_i$ so $\lVert\mathbf{d}_i\rVert=1$).

The rays won't intersect perfectly (noisy patch predictions), so we want the single point $\mathbf{p}$ minimizing the total squared **perpendicular** distance to all rays.

For a candidate point $\mathbf{p}$, decompose $\mathbf{p}-\mathbf{o}_i$ into a component along the ray and one perpendicular to it:
$$
\mathbf{p}-\mathbf{o}_i
= \underbrace{\big(\mathbf{d}_i^{\top}(\mathbf{p}-\mathbf{o}_i)\big)\,\mathbf{d}_i}_{\text{parallel}}
\;+\;
\underbrace{\Big[(\mathbf{p}-\mathbf{o}_i) - \big(\mathbf{d}_i^{\top}(\mathbf{p}-\mathbf{o}_i)\big)\mathbf{d}_i\Big]}_{\text{perpendicular}}.
$$

The parallel part slides freely along the ray (since $t$ is unconstrained), so only the perpendicular part is the true distance. Write it with the projection matrix
$$
M_i = I - \mathbf{d}_i\mathbf{d}_i^{\top}
\qquad(\texttt{np.eye(3) - np.outer(d, d)}).
$$

Then the perpendicular component is $M_i(\mathbf{p}-\mathbf{o}_i)$, and the squared distance to ray $i$ is
$$
\delta_i(\mathbf{p})^2 = \big\lVert M_i(\mathbf{p}-\mathbf{o}_i)\big\rVert^2 .
$$

### Why $M_i = I - \mathbf{d}_i\mathbf{d}_i^{\top}$ is the right operator

$M_i$ projects any vector onto the plane orthogonal to $\mathbf{d}_i$. Two properties make the algebra collapse:

- **Symmetric:** $M_i^{\top} = M_i$.
- **Idempotent:** 
  $$
  M_i^2 = (I - \mathbf{d}_i\mathbf{d}_i^{\top})(I - \mathbf{d}_i\mathbf{d}_i^{\top})
  = I - 2\mathbf{d}_i\mathbf{d}_i^{\top} + \mathbf{d}_i\underbrace{(\mathbf{d}_i^{\top}\mathbf{d}_i)}_{=1}\mathbf{d}_i^{\top}
  = I - \mathbf{d}_i\mathbf{d}_i^{\top} = M_i.
  $$
  (Projecting onto a plane twice does nothing new.)

Using both:
$$
\lVert M_i(\mathbf{p}-\mathbf{o}_i)\rVert^2
= (\mathbf{p}-\mathbf{o}_i)^{\top} M_i^{\top} M_i (\mathbf{p}-\mathbf{o}_i)
= (\mathbf{p}-\mathbf{o}_i)^{\top} M_i (\mathbf{p}-\mathbf{o}_i).
$$

---

## The total cost function

Sum the per-ray squared perpendicular distances over all rays, each counting equally:
$$
E(\mathbf{p}) = \sum_i (\mathbf{p}-\mathbf{o}_i)^{\top} M_i (\mathbf{p}-\mathbf{o}_i).
$$

Each $M_i \succeq 0$ is positive semidefinite, so $E$ is a convex quadratic in $\mathbf{p}$ with a unique minimum wherever the Hessian is positive definite — found by setting the gradient to zero.

---

## Setting the gradient to zero → normal equations

For a quadratic $f(\mathbf{p})=(\mathbf{p}-\mathbf{o})^{\top}M(\mathbf{p}-\mathbf{o})$ with symmetric $M$, the gradient is $\nabla f = 2M(\mathbf{p}-\mathbf{o})$. Summing:
$$
\nabla E(\mathbf{p}) = \sum_i 2\,M_i(\mathbf{p}-\mathbf{o}_i) = \mathbf{0}.
$$

Drop the factor 2 and expand:
$$
\Big(\underbrace{\textstyle\sum_i M_i}_{A}\Big)\mathbf{p}
= \underbrace{\textstyle\sum_i M_i \mathbf{o}_i}_{\mathbf{b}}.
$$

This is exactly the accumulation loop in the triangulation function:

```python
M = np.eye(3) - np.outer(d, d)   # M_i
A += M                           # A = Σ M_i
b += M @ o                       # b = Σ M_i o_i
```

A $3\times3$ linear system $A\mathbf{p}=\mathbf{b}$ — the normal equations. The optimum is
$$
\boxed{\;\mathbf{p}^{\star} = A^{-1}\mathbf{b}\;}
$$
computed with `np.linalg.solve`.

---

## The $+\,\varepsilon I$ regularization

The code actually solves
$$
\mathbf{p}^{\star} = (A + \varepsilon I)^{-1}\mathbf{b}, \qquad \varepsilon = 10^{-9},
$$
i.e. `np.linalg.solve(A + np.eye(3)*1e-9, b)`.

Why: $A = \sum_i (I - \mathbf{d}_i\mathbf{d}_i^{\top})$ — each term is a rank-2 projector that kills direction $\mathbf{d}_i$. $A$ is invertible only if the directions span all of $\mathbb{R}^3$ (rays not all parallel, not all directionally coplanar). If they degenerate — one usable ray, or all directions collinear — $A$ becomes singular and $A^{-1}$ blows up.

Adding $\varepsilon I$ is Tikhonov / ridge regularization: it guarantees $A+\varepsilon I \succ 0$ (strictly positive definite, hence invertible) and bounds the solution norm. At $\varepsilon=10^{-9}$ the bias on a well-conditioned problem is negligible; it only matters in the near-degenerate case, where it returns a stable minimum-norm-ish answer instead of crashing. Formally it minimizes $E(\mathbf{p}) + \varepsilon\lVert\mathbf{p}\rVert^2$.

---

## Residual / reprojection error

After solving, the code measures the fit to each ray:

```python
dists.append(np.linalg.norm(np.cross(p - o, d)))
```

For unit $\mathbf{d}$, the cross-product magnitude **is** the perpendicular distance:
$$
\lVert(\mathbf{p}-\mathbf{o})\times\mathbf{d}\rVert
= \lVert\mathbf{p}-\mathbf{o}\rVert\,\lVert\mathbf{d}\rVert\sin\theta
= \lVert\mathbf{p}-\mathbf{o}\rVert\sin\theta,
$$
the same $\delta_i$ as above, computed via the cross product instead of the projector. Its mean is the triangulation error, used downstream as a confidence signal. (`point_ray_distance` uses the same formula for a single ray.)

---

## The RANSAC wrapper (`ransac_fuse`) — the real outlier defense

Least squares is $L_2$, so one wrong head patch can drag $\mathbf{p}^{\star}$. The dinov3 pipeline doesn't run the solve once on all rays; it embeds it in RANSAC:

1. **Hypothesis generation.** For every pair of distinct views and every combination of their top-$k$ candidate rays, triangulate a 2-ray minimal sample $\to$ a hypothesis point $\mathbf{p}$.
2. **Consensus scoring** (`consensus_for_point`). For each view, pick its single best-matching candidate ray (smallest `point_ray_distance` to $\mathbf{p}$); keep that view as an inlier if the distance is within `threshold`. The threshold is `--ransac-threshold-frac` $\times$ the volume diagonal (default 5%), or an absolute voxel count.
3. **Model selection.** Keep the hypothesis maximizing the number of inlier views, then total inlier score-weight as a tiebreaker: the tuple `(n_inliers, total_weight)`. (The scores still drive view selection here; they just no longer enter the geometric solve.)
4. **Local refinement.** A few refit+reclassify passes (`--ransac-refine-iters`, default 2): refit $\mathbf{p}$ on the current inlier rays via the unweighted triangulation, then re-select inliers. This is iteratively reweighted least squares over the consensus set.

The final $\mathbf{p}^{\star}$ is the unweighted triangulation of the consensus inlier rays. Views outside the consensus are recorded as `outlier_angles`. So the closed-form solve from above is what's evaluated at every RANSAC step and again on the final inlier set — RANSAC just decides *which* rays feed it.

---

## How this relates to "standard" least squares

You can derive the identical system by stacking. The per-ray residual is $\mathbf{r}_i = M_i(\mathbf{p}-\mathbf{o}_i)$. Stacking all of them:
$$
\min_{\mathbf{p}} \sum_i \big\lVert M_i\mathbf{p} - M_i\mathbf{o}_i \big\rVert^2
= \min_{\mathbf{p}} \big\lVert \tilde A\,\mathbf{p} - \tilde{\mathbf{b}} \big\rVert^2,
$$
with $\tilde A = [M_1;\dots;M_n]$ and $\tilde{\mathbf{b}}$ the stacked $M_i\mathbf{o}_i$. The general normal equations $\tilde A^{\top}\tilde A\,\mathbf{p} = \tilde A^{\top}\tilde{\mathbf{b}}$ give
$$
\tilde A^{\top}\tilde A = \sum_i M_i^{\top}M_i = \sum_i M_i = A,
\qquad
\tilde A^{\top}\tilde{\mathbf{b}} = \sum_i M_i\mathbf{o}_i = \mathbf{b},
$$
again using $M_i^{\top}M_i = M_i$. So the accumulate-and-solve form is the normal-equations solution of the stacked weighted least-squares problem — assembled directly as a $3\times3$ system instead of a tall $3n\times3$ matrix.

---

## Conditions and caveats

- **Uniqueness:** needs $\geq 2$ rays with non-parallel directions; `ransac_fuse` returns `None` if fewer than 2 views are usable. The 6 canonical views ($\pm X,\pm Y,\pm Z$) give well-spread directions, so $A$ is well-conditioned.
- **Unweighted solve:** every inlier ray counts equally in the triangulation. Cosine-similarity scores still feed RANSAC (hypothesis ranking and the inlier-tiebreak), but they no longer bias the geometric fit toward higher-confidence patches.
- **Outlier rejection:** RANSAC over the top-$k$ candidates per view is the main defense against a wrong head patch in one or two views — it picks the consensus rather than letting $L_2$ average the error in.
- **Downstream use:** the fused point is clipped to the volume bounds, written as `estimated_head_xyz_voxel` / `_norm`, and the mean ray distance + mean reprojection error are combined into a `confidence` value used by `rotate_head_up.py`.
